In [1]:
import json
import os
import datasets

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to the directory containing the single goal data
single_dir = "/projects/iris/rferreir/datasets/PPNL/original_data/single_goal"

# Path to the directory containing the multiple goal data
multi_dir = "/projects/iris/rferreir/datasets/PPNL/original_data/multi_goal"

# Path to the directory where the files will be saved
output_dir = "/projects/iris/rferreir/GeoBenchmark/data/PPNL"

## Test

In [3]:
def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

In [4]:
def get_coord_obstacles(world):
    obs = []
    start = []
    end = []
    for i in range(len(world)):
        for j in range(len(world[0])):
            if world[i][j] == 1:
                obs.append([i,j])
            elif world[i][j] == 2:
                start.extend([i,j])
            elif world[i][j] == 3:
                end.append([i,j])
    return start, end, obs

In [5]:
def extract_data(path, distribution):
    data = load_json(path)
    final_data = []
    if 'solution_coordinates' not in data[0]:
        print("No solution available. Skip the file.\n")
        return []
    for e in data:
        dic = {}
        dic['matrix_size'] = len(e['world'])
        dic['world_description'] = e['nl_description']
        dic['world'] = e['world']
        start, end, obs = get_coord_obstacles(e['world'])
        dic['obstacles_coords'] = obs
        dic['start'] = start
        dic['end'] = end
        dic['n_goals'] = len(end)
        coords = []
        if e['solution_coordinates'] != "Goal not reachable":
            coords.append([-1, -1])
            for coord in e['solution_coordinates']:
                c = [coord[0], coord[1]]
                if c != coords[-1]:
                    coords.append(c)
            coords.pop(0)
        dic['path'] = coords
        dic['agent_as_a_point'] = e['agent_as_a_point']
        dic['agent_has_direction'] = e['agent_has_direction']
        dic['distribution'] = distribution
        final_data.append(dic)
    #print(final_data[0])
    return final_data

### Single Goal data

In [64]:
final_data_single = {'train': [], 'test': [], 'dev': []}

In [65]:
for file in os.listdir(single_dir):
    if not file.endswith('.json'):
        continue
        
    split = None
    distribution = 'iid'
    if 'test' in file:
        split = 'test'
    elif 'dev' in file:
        split = 'dev'
    else:
        split= 'train'

    if 'unseen' in file:
        split = 'test'
        distribution = 'ood'
        
    print(f"Processing file : {file}")
    final_data_single[split].extend(extract_data(os.path.join(single_dir, file), distribution))

Processing file : 1_goals_test_seen_6x6_samples.json
Processing file : 1_goals_test_unseen_5x5_samples.json
Processing file : 1_goals_test_unseen_6x6more_obstacles_samples.json
Processing file : 1_goals_test_unseen_7x7_samples.json
Processing file : 1_train_set_6x6_samples.json
Processing file : 1dev_set_6x6_samples.json
Processing file : 1goals_unseen_6x6_samples.json


In [66]:
print(len(final_data_single['train']))
print(len(final_data_single['dev']))
print(len(final_data_single['test']))

16032
2004
19044


In [67]:
for split in ['train', 'dev', 'test']:
    with open(os.path.join(output_dir, f"{split}_single.json"), 'w') as f:
        json.dump(final_data_single[split], f)

### Multi Goal data

In [6]:
final_data_multi = {'train': [], 'test': [], 'dev': []}

In [7]:
for n_goals in ['2_goals', '3_goals', '4_goals', '5_goals', '6_goals', 'combined']:
    ndir = os.path.join(multi_dir, n_goals)
    for file in os.listdir(ndir):
        if not file.endswith('.json'):
            continue
        split = None
        distribution = 'iid'
        if 'test' in file:
            split = 'test'
        elif 'dev' in file:
            split = 'dev'
        elif 'train' in file:
            split= 'train'

        if 'unseen' in file:
            split = 'test'
    
        if '5x5' in file or '7x7' in file or 'more_obstacles' in file:
            split = 'test'
            distribution = 'ood'
            
        print(f"Processing file : {file}")
        final_data_multi[split].extend(extract_data(os.path.join(ndir, file), distribution))

Processing file : 2_goals_dev_set_6x6_samples.json
Processing file : 2_goals_test_seen_5x5_samples.json
Processing file : 2_goals_test_seen_6x6_samples.json
Processing file : 2_goals_test_seen_6x6more_obstacles_samples.json
Processing file : 2_goals_test_seen_7x7_samples.json
Processing file : 2_goals_train_set_6x6_samples.json
Processing file : 2_goals_unseen_6x6_samples.json
Processing file : 3_goals_dev_set_6x6_samples.json
Processing file : 3_goals_test_seen_5x5_samples.json
Processing file : 3_goals_test_seen_6x6_samples.json
Processing file : 3_goals_test_seen_6x6more_obstacles_samples.json
Processing file : 3_goals_test_seen_7x7_samples.json
Processing file : 3_goals_train_set_6x6_samples.json
Processing file : 3goals_unseen_6x6_samples.json
Processing file : 4_goals_dev_set_6x6_samples.json
Processing file : 4_goals_test_seen_5x5_samples.json
Processing file : 4_goals_test_seen_6x6_samples.json
Processing file : 4_goals_test_seen_6x6more_obstacles_samples.json
Processing file :

In [8]:
print(len(final_data_multi['train']))
print(len(final_data_multi['dev']))
print(len(final_data_multi['test']))

53440
6680
55440


In [9]:
print(final_data_multi['test'][0])

{'matrix_size': 5, 'world_description': 'You are in a 5 by 5 world. There are obstacles that you have to avoid at: (0,2). You are at (1,1). You have to visit p0 and p1. p0 is located at (0, 3) and p1 is located at (1, 4). ', 'world': [[0, 0, 1, 3, 0], [0, 2, 0, 0, 3], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'obstacles_coords': [[0, 2]], 'start': [1, 1], 'end': [[0, 3], [1, 4]], 'n_goals': 2, 'path': [[1, 1], [1, 2], [1, 3], [0, 3], [0, 4], [1, 4]], 'agent_as_a_point': 'right right up right down ', 'agent_has_direction': 'turn left move forward move forward turn left move forward turn right move forward turn right move forward ', 'distribution': 'ood'}


In [10]:
for split in ['train', 'dev', 'test']:
    with open(os.path.join(output_dir, f"{split}_multi.json"), 'w') as f:
        json.dump(final_data_multi[split], f)

## Converting into HF dataset

In [ ]:
d1_single = datasets.load_dataset('json', data_files={"test":os.path.join(output_dir, "test_single.json"), "train":os.path.join(output_dir, "train_single.json"), "validation":os.path.join(output_dir, "dev_single.json")})

print(d1_single)

#If you want to save the dataset
#d1_single.save_to_disk(os.path.join("/projects/iris/rferreir/hub_datasets", "PPNL_single"))

In [11]:
d1_multi = datasets.load_dataset('json', data_files={"test":os.path.join(output_dir, "test_multi.json"), "train":os.path.join(output_dir, "train_multi.json"), "validation":os.path.join(output_dir, "dev_multi.json")})

print(d1_multi)

#If you want to save the dataset
#d1_multi.save_to_disk(os.path.join("/projects/iris/rferreir/hub_datasets", "PPNL_multi"))

Generating test split: 55440 examples [00:02, 20554.51 examples/s]
Generating train split: 53440 examples [00:02, 20714.68 examples/s]
Generating validation split: 6680 examples [00:00, 20210.32 examples/s]


DatasetDict({
    test: Dataset({
        features: ['matrix_size', 'world_description', 'world', 'obstacles_coords', 'start', 'end', 'n_goals', 'path', 'agent_as_a_point', 'agent_has_direction', 'distribution'],
        num_rows: 55440
    })
    train: Dataset({
        features: ['matrix_size', 'world_description', 'world', 'obstacles_coords', 'start', 'end', 'n_goals', 'path', 'agent_as_a_point', 'agent_has_direction', 'distribution'],
        num_rows: 53440
    })
    validation: Dataset({
        features: ['matrix_size', 'world_description', 'world', 'obstacles_coords', 'start', 'end', 'n_goals', 'path', 'agent_as_a_point', 'agent_has_direction', 'distribution'],
        num_rows: 6680
    })
})


## Sanity Check

In [ ]:

d2_single = datasets.load_dataset("rfr2003/Geo_Benchmark", "PPNL_single")
d2_multi = datasets.load_dataset("rfr2003/Geo_Benchmark", "PPNL_multi")

import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

#It can take some time (1-2 minutes)
for split in ['train', 'test', 'validation']:
    assert content_hash(d1_single[split]) == content_hash(d2_single[split])
    assert content_hash(d1_multi[split]) == content_hash(d2_multi[split])